# Check an Analyst report

Copy this file into `private/reviews/` and rename the copy to `<packet_id>_review.ipynb`
(for example, `EP_0003_review.ipynb`). Work in that copy; keep this template blank.

1. Set `attempt_id` below to the folder name printed by generation.
2. Run the cells and read each claim beside its evidence.
3. Fill in **Notes / feedback** and make any fixes in the edit cell.
4. When finished, set `ready = True`, run all cells again, and save the notebook.

The last cell saves the final report to public. The original candidate stays unchanged.
There are no model calls here. Leave `ready = False` while checking.


In [2]:
import copy
import json
import sys
from decimal import Decimal
from pathlib import Path
from pprint import pprint

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/benchmark/private/packets").is_dir())
sys.path.insert(0, str(repo))
from src.benchmark.prepare import read, save, mask, assert_masked, check_report
from src.benchmark.schemas import Packet, Report

folder = repo / "data/benchmark"
attempt_id = "2f1e45f6e1db42729e9b0995e208ff83"  # change this in your copy
attempt_folder = folder / "private/attempts" / attempt_id
attempt = read(attempt_folder / "attempt.json")
sent_packet = json.loads(attempt["request"]["messages"][1]["content"])["evidence_packet"]
original_report = read(attempt_folder / "candidate.json")
aliases = read(folder / "private/aliases.json")
packet = mask(sent_packet, aliases)
report = copy.deepcopy(original_report)
print("Packet:", packet["packet_id"], "| Model:", attempt["request"]["model"])
print("Packet masking changed:", packet != sent_packet)

Packet: EP_0009 | Model: google/gemini-2.5-flash
Packet masking changed: False


In [3]:
print("Recommendation:", report["recommendation"])
evidence = {e["evidence_id"]: e["text"] for e in packet["evidence"]}
for claim in report["claims"]:
    print("\nCLAIM", claim["claim_id"], claim["text"])
    for evidence_id in claim["evidence_ids"]:
        print("EVIDENCE", evidence_id, evidence.get(evidence_id, "MISSING"))
print("\nRationale:", report["rationale"])

Recommendation: BUY

CLAIM C1 The company's worldwide income before tax significantly increased by 95.25% from $16,687 million in fiscal year 2024 to $32,581 million in fiscal year 2025.
EVIDENCE EP_0009_E01 Income before tax by segment
Income before tax by segment of business was as follows:
 Income Before Tax Segment Sales
Percent of Segment
Sales
(Dollars in Millions) 2025 2024 2025 2024 2025 2024
SEGMENT_15 $ 22,266 18,919 60,401 56,964 36.9 % 33.2 
SEGMENT_16 4,113 3,740 33,792 31,857 12.2 11.7 
Segment earnings before tax 26,379 22,659 94,193 88,821 28.0 25.5 
(Income) Expenses not allocated to segments (6,202) 5,972 
Worldwide income before tax $ 32,581 16,687 94,193 88,821 34.6 % 18.8 
See Note 17 to the Consolidated Financial Statements for more details.
Amounts not allocated to segments include interest (income) expense and general corporate (income) expense. The fiscal year 2025 includes the reversal of approximately $7.0
billion, a significant portion of the previously accr

## Numbers

Compare the claimed amount or percentage with the calculation. Also read the prose above:
these checks only cover the structured assertion, not every number in a sentence.

In [4]:
facts_by_id = {f["fact_id"]: f for f in packet["facts"]}
for claim in report["claims"]:
    spec = claim["check_spec"]
    if spec is None:
        print(claim["claim_id"], "Check against the excerpt above.")
        continue
    ids = [spec.get("fact_id")] if spec["kind"] == "numeric" else [spec.get("from_fact_id"), spec.get("to_fact_id")]
    periods = [spec["period"]] if spec["kind"] == "numeric" else [spec["from_period"], spec["to_period"]]
    if all(ids):
        selected = [facts_by_id.get(fact_id) for fact_id in ids]
        print(claim["claim_id"], "Referenced facts:")
        pprint(selected)
        if any(f is None for f in selected):
            print("Unknown fact ID; check manually.")
            continue
    else:
        print(claim["claim_id"], "Older candidate: references are added in the edit cell below.")
        matches = [[f for f in packet["facts"] if f["metric"] == spec["metric"]
                    and f["evidence_id"] in claim["evidence_ids"] and f["period"] == period
                    and (spec["kind"] != "numeric" or f["unit"] == spec["unit"])] for period in periods]
        if any(len(found) != 1 for found in matches):
            print("No exact fact match; check manually.")
            continue
        selected = [found[0] for found in matches]
        pprint(selected)
    values = [Decimal(f["value"]) for f in selected]
    if spec["kind"] == "percentage_change" and values[0] == 0:
        print(claim["claim_id"], "Cannot calculate percentage change from zero.")
        continue
    expected = values[0] if spec["kind"] == "numeric" else (values[1] - values[0]) / values[0] * 100
    print(claim["claim_id"], "claimed:", spec["value"], "calculated:", expected,
          "rounded:", expected.quantize(Decimal("0.01")))
pprint(check_report(Packet.model_validate(packet), Report.model_validate(report)))


C1 Referenced facts:
[{'evidence_id': 'EP_0009_E01',
  'fact_id': 'EP_0009_F08',
  'metric': 'income_before_tax',
  'period': {'end': '2024-12-29', 'label': 'FY2024', 'start': '2024-01-01'},
  'unit': 'USD_million',
  'value': '16687'},
 {'evidence_id': 'EP_0009_E01',
  'fact_id': 'EP_0009_F07',
  'metric': 'income_before_tax',
  'period': {'end': '2025-12-28', 'label': 'FY2025', 'start': '2024-12-30'},
  'unit': 'USD_million',
  'value': '32581'}]
C1 claimed: 95.25 calculated: 95.24779768682207706597950500 rounded: 95.25
C2 Referenced facts:
[{'evidence_id': 'EP_0009_E01',
  'fact_id': 'EP_0009_F01',
  'metric': 'revenue',
  'period': {'end': '2025-12-28', 'label': 'FY2025', 'start': '2024-12-30'},
  'unit': 'USD_million',
  'value': '94193'}]
C2 claimed: 94193 calculated: 94193 rounded: 94193.00
C3 Check against the excerpt above.
C4 Check against the excerpt above.
C5 Check against the excerpt above.
{'errors': [],
 'human_review': ['C3: check factual support and reasoning manually'

## Notes / feedback

Replace the blanks with short notes. Use the evidence-checking notebook to check the source PDF too.

- Source excerpts and masking: ...
- Claims, numbers, years, units and citations: ...
- Reasoning and risks / counter-evidence: ...
- Problems found and what I changed (or no changes needed): ...
- Decision: not checked yet.


In [5]:
# Make any corrections here. Skip this if no changes are needed.
# Example: report["claims"][0]["text"] = "Corrected claim..."
# If you correct a number, update its check_spec too, if present.
# Keep the edits in this cell so running the notebook again applies them.

# Add explicit source-fact references to the reviewed copy of older candidates.
# This does not change the saved model response, numbers, or wording.
for claim in report["claims"]:
    spec = claim["check_spec"]
    if spec is None:
        continue
    fields = [("fact_id", "period")] if spec["kind"] == "numeric" else [("from_fact_id", "from_period"), ("to_fact_id", "to_period")]
    for id_field, period_field in fields:
        if spec.get(id_field):
            continue
        matches = [f for f in packet["facts"] if f["metric"] == spec["metric"]
                   and f["period"] == spec[period_field] and f["evidence_id"] in claim["evidence_ids"]
                   and (spec["kind"] != "numeric" or f["unit"] == spec["unit"])]
        assert len(matches) == 1, "Check the fact mapping manually: " + claim["claim_id"]
        spec[id_field] = matches[0]["fact_id"]

pprint(report, sort_dicts=False)


{'schema_version': '0.1',
 'report_id': 'R_2f1e45f6e1db42729e9b0995e208ff83',
 'recommendation': 'BUY',
 'claims': [{'claim_id': 'C1',
             'text': "The company's worldwide income before tax significantly "
                     'increased by 95.25% from $16,687 million in fiscal year '
                     '2024 to $32,581 million in fiscal year 2025.',
             'evidence_ids': ['EP_0009_E01'],
             'check_spec': {'kind': 'percentage_change',
                            'from_fact_id': 'EP_0009_F08',
                            'to_fact_id': 'EP_0009_F07',
                            'metric': 'income_before_tax',
                            'from_period': {'label': 'FY2024',
                                            'start': '2024-01-01',
                                            'end': '2024-12-29'},
                            'to_period': {'label': 'FY2025',
                                          'start': '2024-12-30',
                                    

## Save the final copy

Set `ready = True` only when the checks and edits are finished.
This checks the edited report and saves it with the masked packet to `public/`.
It replaces existing files with the same IDs, so use this for the working dataset.
Keep released dataset versions unchanged.

Save this notebook with its outputs. These notes are also the feedback;
there is no separate review JSON or feedback file to fill in.


The report is saved as `public/reports/<packet_id>/clean.json`.
For older candidates, the edit cell adds fact IDs only when metric, period, unit and evidence match uniquely. Check the printed references before saving.

In [6]:
ready = False
if ready:
    final_report = Report.model_validate(report)
    final_packet = Packet.model_validate(packet)
    checks = check_report(final_packet, final_report, require_fact_ids=True)
    assert not checks["errors"], checks["errors"]
    assert_masked(report, aliases)
    assert_masked(packet, aliases)
    save(folder / "public/evidence_packets" / (packet["packet_id"] + ".json"), final_packet.model_dump(mode="json"))
    output = folder / "public/reports" / packet["packet_id"] / "clean.json"
    save(output, final_report.model_dump(mode="json"))
    print("Saved:", output)
else:
    print("Not saved yet.")

Not saved yet.
